[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/barmag/fanous-llm-lens/blob/main/notebooks/education/stage1_c_subword_reference.ipynb)

# Stage 1c: Zero-Layer Transformer (Subword/BPE-Level) 🧩
## target: Is "dialect" a direction you can find in learned embeddings?

We counted characters (Stage 1a) and word transitions (Stage 1b). Now we take the
last step of the embeddings trilogy: instead of *counting*, we **learn a vector for
every subword** by training a tiny model on our own MSA + Masri text — then we ask one
sharp question about those vectors.

### 💡 What we are showing
1. Train a small **BPE tokenizer** on a mixed MSA+Masri corpus (no giant download — the
   subwords come from *our* data).
2. Train a **zero-layer transformer** — literally `embedding → unembedding → softmax over
   the next token`. This is a subword *bigram* predictor, and its embedding matrix `W_E`
   is the thing we inspect.
3. Ask: **is dialect linearly encoded in `W_E`?** We compare two views of the *same*
   vectors —
   - **Unsupervised PCA**, which surfaces the *loudest* variance (frequency/length) and
     leaves the dialect colours mixed, versus
   - **A supervised probe**, which finds the *one axis* that separates MSA from Masri and
     reports an accuracy number.

> ### ⚠️ Up front — what this is *not*
> These are **toy embeddings** from a one-matrix model trained on a small corpus. They
> capture co-occurrence, not deep meaning. Rich contextual semantics only emerge once a
> model has real layers — that is a *later* stage. The lesson here is the method:
> **PCA shows what is biggest; a probe shows what you asked for.**

In [ ]:
# 🛠️ Setup: Install dependencies if running on Google Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q tokenizers datasets scikit-learn plotly pandas torch

## 📚 Step 1 — Build subwords from our own dialect data
We reuse the Stage 1b corpus: **Modern Standard Arabic** from Wikipedia and **Egyptian
(Masri)** from real street tweets. We train a small **BPE** tokenizer on the two combined,
then encode each stream into subword ids. Keeping the two streams separate matters — the
per-stream counts are exactly how we will label each subword's dialect later.

In [ ]:
# 📦 Fetch MSA + Masri text, then learn a BPE vocabulary from it
from datasets import load_dataset
import re
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

print("Fetching MSA from Wikipedia and Masri from Tweets...")
msa_stream = load_dataset("wikimedia/wikipedia", "20231101.ar", split="train", streaming=True)
tweets_ds = load_dataset("amgadhasan/arabic_tweets_dialects", split="train")
eg_tweets = tweets_ds.filter(lambda x: x['dialect'] == 'EG')

def clean_msa(stream, max_chars=500000):
    text = ""
    for article in stream:
        cleaned = re.sub(r'\s+', ' ', article['text'])
        cleaned = re.sub(r'[^\s\u0621-\u064A]', '', cleaned)
        text += cleaned + " "
        if len(text) >= max_chars:
            break
    return text[:max_chars]

def clean_masri(tweets, max_chars=500000):
    text = ""
    for t in tweets:
        cleaned = re.sub(r'\s+', ' ', t['text'])
        cleaned = re.sub(r'[a-zA-Z0-9_@]+', '', cleaned)
        cleaned = re.sub(r'[^\s\u0621-\u064A]', '', cleaned)
        text += cleaned + " "
        if len(text) >= max_chars:
            break
    return text[:max_chars]

print("Collecting MSA...")
msa_text = clean_msa(msa_stream)
print("Collecting Masri...")
masri_text = clean_masri(eg_tweets)

# Train a small BPE vocabulary on the two dialects combined.
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(vocab_size=3000, special_tokens=["[UNK]"])
tokenizer.train_from_iterator([msa_text, masri_text], trainer)

# Encode each stream to subword ids. Shapes: msa_ids/masri_ids -> [n_subwords]
msa_ids = tokenizer.encode(msa_text).ids
masri_ids = tokenizer.encode(masri_text).ids
id_to_token = {i: t for t, i in tokenizer.get_vocab().items()}

print(f"Vocab size: {tokenizer.get_vocab_size()} subwords")
print(f"Encoded {len(msa_ids)} MSA / {len(masri_ids)} Masri subword tokens.")
# Tokenizer-aware logging: a few example fracturings (RTL words shown left-to-right
# as token lists for inspection).
for w in ["الذي", "اللي", "دلوقتي", "الآن", "العربية"]:
    print(f"  {w} -> {tokenizer.encode(w).tokens}")

## 🧠 Step 2 — Train the zero-layer transformer
A **zero-layer transformer** has no attention and no MLP: a token's embedding goes
straight to the unembedding to predict the next token. That makes it a **subword bigram
model**. We train it with cross-entropy on `(current → next)` pairs pooled from both
streams; the learned embedding matrix `W_E` (shape `[vocab, 64]`) is our artefact.

In [ ]:
# 🏋️ Train embedding (W_E) + unembedding (W_U) to predict the next subword
import torch
import numpy as np
import random

torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

EMBED_DIM = 64
EPOCHS = 3
BATCH = 4096
LR = 1e-2

def make_pairs(ids):
    # consecutive (current, next) subword ids within one stream -> [n-1, 2]
    return [(ids[i], ids[i + 1]) for i in range(len(ids) - 1)]

pairs = make_pairs(msa_ids) + make_pairs(masri_ids)  # don't bridge the two streams
assert pairs, (
    f"No bigram pairs (MSA ids: {len(msa_ids)}, Masri ids: {len(masri_ids)}). "
    "Did both dataset fetches return text?"
)
pairs = torch.tensor(pairs, dtype=torch.long)

vocab_size = tokenizer.get_vocab_size()
embed = torch.nn.Embedding(vocab_size, EMBED_DIM)
unembed = torch.nn.Linear(EMBED_DIM, vocab_size, bias=False)
opt = torch.optim.Adam(list(embed.parameters()) + list(unembed.parameters()), lr=LR)
loss_fn = torch.nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    perm = torch.randperm(pairs.shape[0])
    running = 0.0
    n_batches = 0
    for b in range(0, pairs.shape[0], BATCH):
        idx = perm[b:b + BATCH]
        cur, nxt = pairs[idx, 0], pairs[idx, 1]
        logits = unembed(embed(cur))           # [batch, vocab]
        loss = loss_fn(logits, nxt)
        opt.zero_grad()
        loss.backward()
        opt.step()
        running += loss.item()
        n_batches += 1
    avg_loss = running / n_batches if n_batches else float("nan")
    print(f"epoch {epoch}  mean loss {avg_loss:.3f}")

# The learned embeddings. Shape: [vocab, EMBED_DIM]
W_E = embed.weight.detach().numpy()
print(f"Trained W_E shape: {W_E.shape}")

## 🔍 Step 3 — PCA vs. a dialect probe
**Free labels.** Every subword's dialect comes from the data itself: count how often it
appears in the MSA vs Masri stream. Mostly-MSA → `MSA`, mostly-Masri → `Masri`, roughly
even → `Shared` (rare tokens are dropped).

**Two views of the same `W_E`:**
- **PCA** (unsupervised) plots the two directions of largest variance. It does not know
  about dialect, so the colours tend to stay mixed — it shows what is *biggest*.
- **A linear probe** (logistic regression) is *told* the labels and finds the single axis
  that best separates MSA from Masri. We project every subword onto that axis and read off
  a held-out **accuracy**. `Shared` tokens are excluded from the probe entirely — never
  trained or tested on, only projected here, where they should land in the middle.

In [ ]:
# 🎨 Dialect labels, PCA, and a supervised probe — side by side
import numpy as np
from collections import Counter
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from plotly.subplots import make_subplots
import plotly.graph_objects as go

COLORS = {"MSA": "red", "Masri": "blue", "Shared": "green"}

def dialect_labels(msa_ids, masri_ids, min_count=5, hi=0.7, lo=0.3):
    """Label each subword by which stream it favours. -> {id: 'MSA'|'Masri'|'Shared'}"""
    cm, cs = Counter(msa_ids), Counter(masri_ids)
    labels = {}
    for tok in set(cm) | set(cs):
        m, s = cm.get(tok, 0), cs.get(tok, 0)
        total = m + s
        if total < min_count:
            continue
        share = m / total           # fraction of appearances in the MSA stream
        if share >= hi:
            labels[tok] = "MSA"
        elif share <= lo:
            labels[tok] = "Masri"
        else:
            labels[tok] = "Shared"
    return labels

def probe_dialect(W_E, labels, seed=0, test_size=0.3):
    """Logistic probe: can a single linear axis separate MSA from Masri?"""
    bin_ids = [t for t, l in labels.items() if l in ("MSA", "Masri")]
    X = W_E[bin_ids]
    y = np.array([1 if labels[t] == "MSA" else 0 for t in bin_ids])
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=seed, stratify=y
    )
    clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    direction = clf.coef_[0]                       # the dialect axis, shape [dim]
    projections = {t: float(W_E[t] @ direction) for t in labels}
    boundary = -float(clf.intercept_[0])           # w·x = -b at the decision boundary
    return {
        "accuracy": clf.score(X_te, y_te),
        "direction": direction,
        "projections": projections,
        "boundary": boundary,
    }

def run_pca(W_E, ids, seed=0):
    """2-component PCA of the selected rows. -> [len(ids), 2]"""
    return PCA(n_components=2, random_state=seed).fit_transform(W_E[ids])

def plot_pca_vs_probe(W_E, labels, probe, id_to_token):
    ids = list(labels.keys())
    coords = run_pca(W_E, ids)
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("PCA (unsupervised) — mixed",
                        f"Dialect probe axis — acc {probe['accuracy']:.2f}"),
    )
    # Left: PCA scatter, coloured by dialect.
    for dia in ("MSA", "Masri", "Shared"):
        rows = [k for k, t in enumerate(ids) if labels[t] == dia]
        if not rows:
            continue
        fig.add_trace(
            go.Scatter(
                x=coords[rows, 0], y=coords[rows, 1], mode="markers",
                marker=dict(color=COLORS[dia], size=6), name=dia,
                text=[id_to_token.get(ids[k], str(ids[k])) for k in rows],
            ),
            row=1, col=1,
        )
    # Right: histogram of probe-axis projections, per dialect.
    for dia in ("MSA", "Masri", "Shared"):
        vals = [probe["projections"][t] for t in ids if labels[t] == dia]
        if not vals:
            continue
        fig.add_trace(
            go.Histogram(x=vals, marker=dict(color=COLORS[dia]), opacity=0.6,
                         name=dia, showlegend=False),
            row=1, col=2,
        )
    fig.add_vline(x=probe["boundary"], line=dict(color="white", dash="dash"),
                  row=1, col=2)
    fig.update_layout(barmode="overlay", width=950, height=480,
                      title="Learned subword embeddings: PCA can't see dialect, a probe can")
    return fig

# Driver
labels = dialect_labels(msa_ids, masri_ids)
# Drop the [UNK] catch-all: no dialect meaning, only clutters the picture.
labels = {t: l for t, l in labels.items() if id_to_token.get(t) != "[UNK]"}
probe = probe_dialect(W_E, labels)
print(f"Held-out dialect-probe accuracy: {probe['accuracy']:.2f}  "
      f"({sum(v=='MSA' for v in labels.values())} MSA / "
      f"{sum(v=='Masri' for v in labels.values())} Masri / "
      f"{sum(v=='Shared' for v in labels.values())} Shared subwords)")
fig = plot_pca_vs_probe(W_E, labels, probe, id_to_token)
fig.show()

## ✅ Recap & hand-off
- We **learned** a vector per subword by training a one-matrix (zero-layer) model on our
  own MSA + Masri text — completing the trilogy: *count chars → count words → learn vectors*.
- **PCA** of those vectors leaves the dialects mixed: the loudest variance is frequency and
  length, not dialect.
- A **linear probe**, told the labels, recovers a dialect axis with clear held-out
  accuracy — so **dialect is linearly encoded even in a zero-layer model trained on mixed
  data**. The takeaway is the method: *PCA shows what is biggest; a probe shows what you
  asked for.*

**Next:** with real attention + MLP layers, embeddings start carrying *meaning*, not just
co-occurrence. That is where contextual semantics — and the MSA↔Masri story inside the
residual stream — pick up in the next stage.